# ToN-IoT Network — thí nghiệm 1 triệu dòng

Notebook này chỉ gọi chương trình trong `src/ton_iot`. Không chạy tự động cả ba mô hình. Hãy điền URL repository và đường dẫn sample trên Google Drive trước khi chạy.

In [ ]:
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

try:
    import psutil
    print(f'Python: {sys.version}')
    print(f'CPU logical: {psutil.cpu_count(logical=True)}')
    print(f'RAM total GiB: {psutil.virtual_memory().total / 1024**3:.2f}')
    print(f'RAM available GiB: {psutil.virtual_memory().available / 1024**3:.2f}')
except ImportError:
    print(f'Python: {sys.version}')
print(f'Platform: {platform.platform()}')
print(f'Disk free GiB: {shutil.disk_usage("/content").free / 1024**3:.2f}')
gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True, check=False)
print('nvidia-smi:', 'available' if gpu.returncode == 0 else 'not available')
if gpu.returncode == 0:
    print(gpu.stdout[:2000])

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/ton_iot_network')
PROJECT_DIR = Path('/content/ton_iot_network')
DRIVE_SAMPLE_DIR = DRIVE_ROOT / 'network_1m'
LOCAL_SAMPLE_DIR = PROJECT_DIR / 'colab_data' / 'network_1m'
DRIVE_RESULT_DIR = DRIVE_ROOT / 'results' / 'network_1m'
print('Drive sample:', DRIVE_SAMPLE_DIR)
print('Drive results:', DRIVE_RESULT_DIR)

In [ ]:
GITHUB_REPO_URL = ''  # Điền URL repository trước khi chạy cell này.
GITHUB_BRANCH = 'refactor/unified-src'
if not GITHUB_REPO_URL:
    raise ValueError('Hãy điền GITHUB_REPO_URL; notebook không ghi cứng URL GitHub.')
if PROJECT_DIR.exists():
    print('Project directory already exists:', PROJECT_DIR)
else:
    subprocess.run(['git', 'clone', '--branch', GITHUB_BRANCH, GITHUB_REPO_URL, str(PROJECT_DIR)], check=True)
print('Project:', PROJECT_DIR)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_DIR / 'requirements.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT_DIR)], check=True)
print('Dependencies and editable package installed.')

In [ ]:
if not (DRIVE_SAMPLE_DIR / 'sampling_report.json').is_file():
    raise FileNotFoundError(f'Missing sample: {DRIVE_SAMPLE_DIR}')
LOCAL_SAMPLE_DIR.parent.mkdir(parents=True, exist_ok=True)
if not LOCAL_SAMPLE_DIR.exists():
    shutil.copytree(DRIVE_SAMPLE_DIR, LOCAL_SAMPLE_DIR)
print('Local sample ready:', LOCAL_SAMPLE_DIR)
print('Disk free after copy GiB:', shutil.disk_usage('/content').free / 1024**3)

In [ ]:
env = os.environ.copy()
env['PYTHONPATH'] = str(PROJECT_DIR / 'src')
validation_report = DRIVE_SAMPLE_DIR / 'colab_validation_report.json'
subprocess.run([sys.executable, '-m', 'ton_iot.training.experiments', 'validate', '--data-dir', str(LOCAL_SAMPLE_DIR), '--report', str(validation_report)], cwd=PROJECT_DIR, env=env, check=True)
print('Sample validation completed:', validation_report)

In [ ]:
RUN_GPU_CHECK = False
if RUN_GPU_CHECK:
    subprocess.run([sys.executable, '-m', 'ton_iot.training.train', '--experiment', 'network_1m', '--model', 'xgboost', '--gpu-check', '--device', 'gpu', '--data-dir', str(LOCAL_SAMPLE_DIR), '--cpu-threads', '2'], cwd=PROJECT_DIR, env=env, check=True)
else:
    print('GPU check skipped. Set RUN_GPU_CHECK=True only on a Colab runtime with T4 and nvidia-smi.')

In [ ]:
MODEL = 'rf'  # Chọn đúng một: rf, xgboost, lightgbm
DEVICE = 'cpu'
MAX_RAM_GIB = 10.0
if MODEL not in {'rf', 'xgboost', 'lightgbm'}:
    raise ValueError('MODEL must be rf, xgboost or lightgbm')
result_dir = DRIVE_RESULT_DIR / MODEL
command = [sys.executable, '-m', 'ton_iot.training.train', '--experiment', 'network_1m', '--model', MODEL, '--train-mode', 'full', '--data-dir', str(LOCAL_SAMPLE_DIR), '--output-dir', str(result_dir), '--device', DEVICE, '--cpu-threads', '2', '--max-ram-gib', str(MAX_RAM_GIB)]
print('Running:', ' '.join(command))
subprocess.run(command, cwd=PROJECT_DIR, env=env, check=True)
print('Model results saved to:', result_dir)

## Lưu ý

Sau khi từng model hoàn tất, kiểm tra `run_manifest.json`, `metrics.csv`, `attack_type_results.csv` và file log trong thư mục kết quả trên Drive. Muốn chạy model khác, đổi `MODEL` rồi chạy lại cell huấn luyện; không dùng `all`.